## Wstęp do Uczenia Maszynowego

#### Laboratorium 07


### Zakres laboratorium

- regularyzacja Elastic Net w regresji logistycznej (z wykorzystaniem `Pipeline` i `ColumnTransformer` z lab06)
- maszyny wektorów podpierających (SVM)
- margines, wektory podpierające i parametr `C`
- dobór parametru `C` metodą GridSearchCV


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
import sklearn.model_selection as skm

### 1. Regresja logistyczna z regularyzacją Elastic Net

Elastic Net łączy kary L1 i L2 w jednej funkcji kosztu:

$$
\mathcal{L}(\boldsymbol{\beta})
= -\frac{1}{n} \sum_{i=1}^{n}
\Big[
y_i \log(p_i) + (1 - y_i) \log(1 - p_i)
\Big]
+ \lambda \left(
\alpha \|\boldsymbol{\beta}\|_1 +
\frac{1 - \alpha}{2} \|\boldsymbol{\beta}\|_2^2
\right).
$$

Parametr $\alpha \in [0, 1]$ określa udział kary L1: dla $\alpha=1$ otrzymujemy Lasso, dla $\alpha=0$ Ridge, a dla wartości pośrednich — Elastic Net. W `scikit-learn` odpowiada mu parametr `l1_ratio`.


---
##### *Zadanie 1*

Na danych `credit-g`:

 a) Wczytaj dane, wybierz zmienne `age`, `existing_credits`, `duration`, `credit_amount`, `savings_status` i `credit_history`, a klasę `good` zakoduj jako 1.

 b) Podziel dane na zbiory treningowy i testowy. Przygotowanie danych połącz z modelem w `Pipeline`, tak jak w lab06: `ColumnTransformer` powinien standaryzować zmienne numeryczne (`StandardScaler`) i kodować zmienne kategoryczne (`OneHotEncoder`).

 c) Dla każdej wartości
$$C \in \{0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10, 50, 100\}$$
oceń pipeline z regresją logistyczną z karą L1 za pomocą 5-krotnej walidacji krzyżowej na zbiorze treningowym (`cross_val_score`, miara `roc_auc`). Wyznacz też liczbę niezerowych współczynników modelu dopasowanego do całego zbioru treningowego. Jak zmienia się liczba niezerowych współczynników wraz ze wzrostem `C`? Dlaczego dla najmniejszych wartości `C` AUC wynosi 0.5?

 d) Ustal `C = 0.1` i w ten sam sposób porównaj modele Elastic Net dla różnych wartości `l1_ratio`. Wybierz `l1_ratio`, uzasadnij wybór, a następnie dopasuj wybrany model do całego zbioru treningowego i raz oceń AUC na zbiorze testowym.

Dzięki `Pipeline` skaler i koder są w każdym foldzie dopasowywane wyłącznie do jego części uczącej, więc ani fold walidacyjny, ani zbiór testowy nie wpływają na przygotowanie danych.


### 2. Maszyny wektorów podpierających (SVM)


Gdy klasy da się rozdzielić prostą (w ogólności: hiperpłaszczyzną) $\langle w, x\rangle + b = 0$, takich prostych jest zazwyczaj nieskończenie wiele (rysunek poniżej, po lewej). **Klasyfikator maksymalnego marginesu** wybiera tę, która jest najdalej od najbliższych obserwacji obu klas (po prawej). Odległość od prostej do tych obserwacji nazywamy **marginesem**, a obserwacje leżące na granicy marginesu — **wektorami podpierającymi**. Tylko one wyznaczają położenie prostej: przesunięcie pozostałych punktów (bez przekraczania marginesu) nie zmienia modelu.

![](svm_1.png)

W praktyce klasy rzadko są idealnie rozdzielne, dlatego SVM dopuszcza naruszenia marginesu, wprowadzając zmienne $\xi_i \ge 0$. W `scikit-learn` (`SVC`) rozwiązywane jest zadanie

$$
\min_{w,\, b,\, \xi}\ \frac{1}{2}\|w\|^2 + C \sum_{i=1}^{n} \xi_i
\quad \text{przy warunkach} \quad
y_i\left(\langle w, x_i\rangle + b\right) \ge 1 - \xi_i,\quad \xi_i \ge 0,
$$

gdzie $y_i \in \{-1, 1\}$. Granice marginesu to proste $\langle w, x\rangle + b = \pm 1$, odległe od prostej rozdzielającej o $\frac{1}{\|w\|}$. Wektorami podpierającymi są teraz wszystkie obserwacje leżące na granicy marginesu, wewnątrz niego lub po złej stronie prostej.

Parametr `C` to **kara za naruszenia marginesu**:
- duże `C` — naruszenia są kosztowne, margines jest wąski, mało wektorów podpierających (ryzyko przeuczenia);
- małe `C` — naruszenia są tanie, margines jest szeroki, wiele wektorów podpierających (większe obciążenie, mniejsza wariancja).


Do rysowania dopasowanych modeli wykorzystamy pomocniczą funkcję `plot_svm_2d`.


In [ ]:
# Uproszczona adaptacja ISLP.svm.plot, umieszczona lokalnie, aby uniknąć
# instalowania pakietu ISLP wraz z rozbudowanymi zależnościami.
# Źródło: https://github.com/intro-stat-learning/ISLP/blob/main/ISLP/svm.py
def plot_svm_2d(X, y, model, ax=None, grid_size=300):
    """Rysuje obszary decyzyjne, margines i wektory podpierające liniowego SVM."""
    X = np.asarray(X)
    y = np.asarray(y)
    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError("Funkcja wymaga danych z dokładnie dwiema zmiennymi.")

    if ax is None:
        _, ax = plt.subplots()

    x0, x1 = X[:, 0], X[:, 1]
    xlim = (x0.min() - 0.5 * x0.std(), x0.max() + 0.5 * x0.std())
    ylim = (x1.min() - 0.5 * x1.std(), x1.max() + 0.5 * x1.std())
    xx, yy = np.meshgrid(
        np.linspace(*xlim, grid_size), np.linspace(*ylim, grid_size)
    )
    grid = np.c_[xx.ravel(), yy.ravel()]

    ax.scatter(x0, x1, c=y, cmap=plt.cm.tab10, s=200)
    ax.contourf(
        xx, yy, model.predict(grid).reshape(xx.shape),
        cmap=plt.cm.plasma, alpha=0.2
    )
    decision = model.decision_function(grid).reshape(xx.shape)
    ax.contourf(xx, yy, decision, levels=[-1, 1], cmap=plt.cm.plasma, alpha=0.2)
    support_idx = model.support_
    support_decision = model.decision_function(X[support_idx])
    signed_y = np.where(y[support_idx] == model.classes_[1], 1, -1)
    correctly_classified = signed_y * support_decision > 0
    ax.scatter(
        X[support_idx, 0][correctly_classified],
        X[support_idx, 1][correctly_classified],
        marker="+", c="black", s=80, linewidths=1.5
    )
    ax.scatter(
        X[support_idx, 0][~correctly_classified],
        X[support_idx, 1][~correctly_classified],
        marker="x", c="red", s=80, linewidths=1.5
    )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    return ax

In [ ]:
# losowanie dwóch zmiennych z rozkładu standardowego normalnego
rng = np.random.default_rng(1)
X = rng.standard_normal((50,2))
y = np.array([-1]*25 + [1]*25)
# przesunięcie punktów o wektor [2, 2]
X[y == 1] += 2

In [ ]:
# wykres punktów
plt.scatter(X[:, 0], X[:, 1], c=y)
plt.show()

-----
##### *Zadanie 2*

Zbuduj model wektorów podpierających, wykorzystując zdefiniowane `X` i `y` jako zbiór treningowy. Ustaw parametr `C = 10` oraz `kernel = 'linear'`. Narysuj dopasowany model funkcją `plot_svm_2d`.


**Jak czytać wykres `plot_svm_2d`:**
- kolor tła — obszar decyzyjny, czyli klasa, którą model przypisuje punktom z danego obszaru;
- ciemniejszy pas wzdłuż granicy — margines, czyli obszar między prostymi $\langle w, x\rangle + b = -1$ i $\langle w, x\rangle + b = 1$;
- `+` — wektor podpierający sklasyfikowany poprawnie, ale leżący na granicy marginesu lub wewnątrz niego;
- `x` — błędnie sklasyfikowany wektor podpierający.


-----
##### *Zadanie 3*

Dla modelu z Zadania 2 narysuj samodzielnie, bez funkcji `plot_svm_2d`, prostą rozdzielającą klasy i wektory podpierające.

 a) Odczytaj wektor $w$ (atrybut `coef_`) i wyraz wolny $b$ (atrybut `intercept_`), a następnie wyznacz współczynniki $A$ i $B$ prostej rozdzielającej (wzory poniżej).

 b) Wyznacz, które punkty są wektorami podpierającymi.

 c) Narysuj punkty, prostą rozdzielającą dla $x_1 \in [-1.5, 3]$ oraz zaznacz wektory podpierające.

#### Wyznaczanie hiperpłaszczyzny (dla dwóch wymiarów)

$\langle w, x\rangle + b = 0$

$w_1x_1 + w_2x_2 + b = 0$

$w_2x_2 = -w_1x_1 - b$

$x_2 = -\frac{w_1}{w_2}x_1 - \frac{b}{w_2} = A x_1 + B$


In [ ]:
W = ...  # wektor w (coef_)
b = ...  # wyraz wolny (intercept_)

A = ...
B = ...

-----
##### *Zadanie 4*

Zbuduj model wektorów podpierających, wykorzystując zdefiniowane `X` i `y` jako zbiór treningowy. Ustaw parametr `C = 0.01` oraz `kernel = 'linear'`.

 a) Narysuj dopasowany model funkcją `plot_svm_2d`.

 b) Tak jak w Zadaniu 3 wyznacz współczynniki $A$ i $B$ prostej rozdzielającej.

 c) Wyznacz wektor jednostkowy $\hat{w}$ oraz szerokość marginesu $m$ (wzory poniżej).

 d) Narysuj punkty, prostą rozdzielającą oraz granice marginesu. Granice marginesu otrzymamy, przesuwając prostą rozdzielającą o $\pm\, m \hat{w}$ (kod poniżej).

#### Wyznaczanie marginesu (dla dwóch wymiarów)

$m = \frac{1}{\|w\|}$

$\|w\| = \sqrt{w_1^2 + w_2^2}$

$\hat{w} = \frac{w}{\|w\|}$


In [ ]:
W = ...  # wektor w (coef_)
b = ...  # wyraz wolny (intercept_)

A = ...
B = ...

In [ ]:
w_hat = ...

In [ ]:
margin = ...

In [ ]:
decision_boundary_points = np.array(list(zip(np.array([-1.5, 3]), A * np.array([-1.5, 3]) + B)))
points_of_line_above = decision_boundary_points + w_hat * margin
points_of_line_below = decision_boundary_points - w_hat * margin

In [ ]:
fig, ax = plt.subplots()
ax.scatter(X[:, 0], X[:, 1], c=y)
ax.plot(np.array([-1.5, 3]), A * np.array([-1.5, 3]) + B)
# granica marginesu po stronie klasy 1 (przesunięcie w kierunku w)
ax.plot(points_of_line_above[:, 0],
        points_of_line_above[:, 1],
        'b--',
        linewidth=2)
# granica marginesu po stronie klasy -1 (przesunięcie przeciwnie do w)
ax.plot(points_of_line_below[:, 0],
        points_of_line_below[:, 1],
        'g--',
        linewidth=2)
plt.show()

#### Optymalizacja parametru C z wykorzystaniem GridSearchCV


In [ ]:
kfold = skm.StratifiedKFold(5,
                            random_state=0,
                            shuffle=True)

In [ ]:
grid = skm.GridSearchCV(SVC(kernel='linear'),
                        {'C': [0.001, 0.01, 0.1, 1, 5, 10, 100]},
                        refit=True,  # dopasowanie modelu do całych danych treningowych po wybraniu optymalnych hiperparametrów
                        cv=kfold,  # informacje o strategii kroswalidacji
                        scoring='accuracy')

-----
##### *Zadanie 5*

 a) Dopasuj `grid` do danych `X`, `y`. Wypisz najlepsze hiperparametry (`best_params_`) i przedstaw wyniki walidacji krzyżowej (`cv_results_`) w postaci `DataFrame`, ograniczając się do wartości `C`, średniej i odchylenia standardowego dokładności oraz rankingu.

 b) Wygeneruj zbiór testowy (kod poniżej) i oceń na nim najlepszy model (`best_estimator_`): wyznacz macierz pomyłek i dokładność.


In [ ]:
# zbiór testowy z tego samego rozkładu co dane treningowe
rng = np.random.default_rng(2)
X_test_svm = rng.standard_normal((20, 2))
y_test_svm = np.array([-1]*10 + [1]*10)
X_test_svm[y_test_svm == 1] += 2  # przesunięcie punktów z klasy 1 o [2, 2]

### Podsumowanie, czyli co warto wiedzieć...

1. Elastic Net łączy regularyzację L1 i L2, a parametr `l1_ratio` określa ich proporcję.
2. Przygotowanie danych łączymy z modelem w `Pipeline`, aby transformacje były dopasowywane wyłącznie do danych uczących.
3. SVM szuka granicy rozdzielającej klasy o możliwie szerokim marginesie.
4. Wektory podpierające to obserwacje wyznaczające położenie tej granicy.
5. Parametr `C` określa kompromis między szerokim marginesem a liczbą błędów treningowych.
6. Hiperparametry dobieramy na zbiorze walidacyjnym lub walidacją krzyżową na danych treningowych — nigdy na zbiorze testowym.
